In [1]:
#Notebook formatting
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
print('complete')

complete


In [3]:
sns.set_style("whitegrid")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
#pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
%load_ext memory_profiler

In [5]:
%load_ext autotime

time: 50.2 μs (started: 2026-06-24 20:04:34 -05:00)


In [ ]:
#Note - DFR load attempts that wont work:
#dfr_data = pq.read_table('LCR_cleaned_final.parquet', nrows=1_000_000)
#dfr = dfr_data.to_pandas().sample(100000, random_state=42)
#dfr = pd.read_parquet('LCR_cleaned_final.parquet')
#dfr_sample = dfr.sample(n=100000, random_state=42)

#dfr_data = pq.ParquetFile('LCR_cleaned_final.parquet')
#batch = next(pf.iter_batches(batch_size=1_000_000))
#dfr = pa.Table.from_batches([batch]).to_pandas().sample(100000, random_state=42)

#dataset = pq.ParquetDataset('LCR_cleaned_final')
#table = dataset.read_pandas(use_threads=True)
#dfr = table.to_pandas().head(1000000)

In [ ]:
# This cell tests whether our dfr sample is representative of the full dataset.
# Run this cell first, compare the .describe() outputs, then comment out this cell and the cell underneath once a good sample size is confirmed.

# Reason: The full LCR_cleaned_final dataset is very large (~9GB, 27.6M rows). I used a smaller random sample for faster analysis and statistical representativeness.

risk_score_pop = pd.read_parquet('LCR_cleaned_final', columns=['risk_score', 'dti'])
risk_score_sample = risk_score_pop.sample(500000, random_state=42)
print('population description')
print(risk_score_pop.describe())
print(f"Row Count: {len(risk_score_pop)}")
print(f"   risk_score_pop memory usage: {risk_score_pop.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

In [ ]:
print('sample description')
print(risk_score_sample.describe())
del risk_score_pop
del risk_score_sample

In [6]:
print("Loading important columns for quick testing.")
cols = [
    'addr_state',
'annual_inc',
'delinq_2yrs',
'dti',
'emp_length_lt1',
'emp_length',
'fico_range_high',
'fico_range_low',
'home_ownership',
'inq_last_6mths',
'int_rate',
'issue_d',
'last_fico_range_high',
'last_fico_range_low',
'loan_amnt',
'loan_status',
'pub_rec',
'purpose',
'revol_bal',
'revol_util',
'sub_grade',
'term',
'verification_status',
]
dfa_data = pd.read_parquet('LCA_cleaned_final', columns=cols) #2,260,701
dfa = pd.read_parquet('LCA_cleaned_final', columns=cols).head(500000)  
#dfa = dfa_data.sample(100000, random_state=42)


dfr_data = ds.dataset('LCR_cleaned_final').scanner().head(1000).to_pandas() #Currently at 1k for basic analysis and saving RAM. Set to 500k for a correctly representative sample. 
#dfr = dfr_data.sample(500000, random_state=42)


#test = pd.read_parquet('LCR_cleaned_final') #27,648,741 rows. Appx ~9GB
print("loaded")

Loading important columns for quick testing.
loaded
time: 1.47 s (started: 2026-06-24 20:04:39 -05:00)


### Quick exploration

In [7]:
print(f"Loaded dfa: {dfa_data.shape[0]:,} rows, {dfa_data.shape[1]} columns")
print("Initial memory usage:")
print(f"   dfa: {dfa_data.memory_usage(deep=True).sum() / (1024**3):.1f} GB")
print(f"   dfr: {dfr_data.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

Loaded dfa: 2,260,701 rows, 23 columns
Initial memory usage:
   dfa: 1.0 GB
   dfr: 0.2 MB
time: 1.04 s (started: 2026-06-24 20:04:40 -05:00)


In [27]:
dfa_data.head(5)

,addr_state,annual_inc,delinq_2yrs,dti,emp_length_lt1,emp_length,fico_range_high,fico_range_low,home_ownership,inq_last_6mths,int_rate,issue_d,last_fico_range_high,last_fico_range_low,loan_amnt,loan_status,pub_rec,purpose,revol_bal,revol_util,sub_grade,term,verification_status,delinquency_tf,defaulted_tf
__null_dask_index__,,,,,,,,,,,,,,,,,,,,,,,,,
0,PA,55000.0,0,5.91,0,10,679.0,675.0,MORTGAGE,1,13.99,2015-12-01,564,560,3600.0,Fully Paid,0,debt_consolidation,2765.0,29.700001,C4,36,Not Verified,0,0
1,SD,65000.0,1,16.059999,0,10,719.0,715.0,MORTGAGE,4,11.99,2015-12-01,699,695,24700.0,Fully Paid,0,small_business,21470.0,19.200001,C1,36,Not Verified,0,0
2,IL,63000.0,0,10.78,0,10,699.0,695.0,MORTGAGE,0,10.78,2015-12-01,704,700,20000.0,Fully Paid,0,home_improvement,7869.0,56.200001,B4,60,Not Verified,0,0
3,NJ,110000.0,0,17.059999,0,10,789.0,785.0,MORTGAGE,0,14.85,2015-12-01,679,675,35000.0,Current,0,debt_consolidation,7802.0,11.6,C5,60,Source Verified,0,0
4,PA,104433.0,1,25.370001,0,3,699.0,695.0,MORTGAGE,3,22.450001,2015-12-01,704,700,10400.0,Fully Paid,0,major_purchase,21929.0,64.5,F1,60,Source Verified,0,0


time: 16.6 ms (started: 2026-06-24 21:48:48 -05:00)


In [ ]:
dfr_data.head(5)

In [ ]:
dfa_data.info()

In [ ]:
dfr_data.info()

In [ ]:
dfa_data.describe().round(2)

In [ ]:
dfr_data.describe().round(2)

### Creating cohort framework and more testing

In [8]:
status_cts = dfa_data['loan_status'].value_counts()
status_p = dfa_data['loan_status'].value_counts(normalize=True)
by_year_cts = dfa_data['issue_d'].dt.year.value_counts()
by_year_p = dfa_data['issue_d'].dt.year.value_counts(normalize=True)
by_purpose_cts = dfa_data['purpose'].value_counts()
by_purpose_p = dfa_data['purpose'].value_counts(normalize=True)
by_state_cts = dfa_data['addr_state'].value_counts()
by_state_p = dfa_data['addr_state'].value_counts(normalize=True)
by_grade_cts = dfa_data['sub_grade'].value_counts()
by_grade_p = dfa_data['sub_grade'].value_counts(normalize=True)

dlqcy = ['Charged Off','Late (31-120 days)','Late (16-30 days)','Default']
delinquencies = dfa_data['loan_status'].isin(dlqcy).value_counts()
delinquencies_p = dfa_data['loan_status'].isin(dlqcy).value_counts(normalize=True)
delinquencies_df = dfa_data[dfa_data['loan_status'].isin(dlqcy)]

dfa_data['delinquency_tf'] = dfa_data['loan_status'].isin(dlqcy).astype(int)
dflt = ['Charged Off','Default']
dfa_data['defaulted_tf'] = dfa_data['loan_status'].isin(dflt).astype(int)


dfa['defaulted_tf'] = dfa['loan_status'].isin(dflt).astype(int)
dfa['delinquency_tf'] = dfa['loan_status'].isin(dlqcy).astype(int)

time: 1.32 s (started: 2026-06-24 20:04:45 -05:00)


In [10]:
delinquency_by_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].value_counts()

time: 144 ms (started: 2026-06-24 20:06:06 -05:00)


In [9]:
#Creating delinquency rate/proportion cohorts by:
# Sub grade
# Issuance date
# Purpose
# State
# Employment length of one year or less.
# Employment Length
# Debt-to-Income ratio


delinquency_rate_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].mean() 
delinquency_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].mean() 
delinquency_rate_purpose = dfa_data.groupby('purpose')['delinquency_tf'].mean() 
delinquency_rate_state = dfa_data.groupby('addr_state')['delinquency_tf'].mean() 
delinquency_rate_low_employment = dfa_data.groupby('emp_length_lt1')['delinquency_tf'].mean() 
delinquency_rate_emp_length = dfa_data.groupby('emp_length')['delinquency_tf'].mean() 
delinquency_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['delinquency_tf'].mean()

default_rate_subgrade = dfa_data.groupby('sub_grade')['defaulted_tf'].mean() 
default_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['defaulted_tf'].mean() 
default_rate_purpose = dfa_data.groupby('purpose')['defaulted_tf'].mean() 
default_rate_state = dfa_data.groupby('addr_state')['defaulted_tf'].mean() 
default_rate_low_employment = dfa_data.groupby('emp_length_lt1')['defaulted_tf'].mean() 
default_rate_emp_length = dfa_data.groupby('emp_length')['defaulted_tf'].mean() 
default_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['defaulted_tf'].mean()

time: 1min 16s (started: 2026-06-24 20:04:48 -05:00)


In [30]:
print(delinquency_rate_dti, default_rate_dti)

dti
(-2.0, 99.0]      0.130345
(99.0, 199.0]     0.081218
(199.0, 299.0]    0.079772
(299.0, 399.0]    0.085106
(399.0, 499.0]    0.058824
(499.0, 599.0]    0.080000
(599.0, 699.0]    0.060606
(699.0, 799.0]    0.086957
(799.0, 899.0]    0.142857
(899.0, 999.0]    0.083333
Name: delinquency_tf, dtype: float64 dti
(-2.0, 99.0]      0.118946
(99.0, 199.0]     0.055274
(199.0, 299.0]    0.054131
(299.0, 399.0]    0.049645
(399.0, 499.0]    0.035294
(499.0, 599.0]    0.060000
(599.0, 699.0]    0.030303
(699.0, 799.0]    0.086957
(799.0, 899.0]    0.142857
(899.0, 999.0]    0.062500
Name: defaulted_tf, dtype: float64
time: 1.86 ms (started: 2026-06-24 21:58:27 -05:00)


In [11]:
delinquency_rate_year

issue_d
2007.0    0.074627
2008.0    0.103218
2009.0    0.112479
2010.0    0.118609
2011.0    0.151789
2012.0    0.161973
2013.0    0.155970
2014.0    0.176417
2015.0    0.183906
2016.0    0.169657
2017.0    0.109710
2018.0    0.035736
Name: delinquency_tf, dtype: float64

time: 1.95 ms (started: 2026-06-24 20:06:09 -05:00)


In [12]:
default_rate_year

issue_d
2007.0    0.074627
2008.0    0.103218
2009.0    0.112479
2010.0    0.118609
2011.0    0.151789
2012.0    0.161973
2013.0    0.155948
2014.0    0.174690
2015.0    0.180016
2016.0    0.157115
2017.0    0.088302
2018.0    0.017919
Name: defaulted_tf, dtype: float64

time: 2.04 ms (started: 2026-06-24 20:06:09 -05:00)


In [13]:
dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].value_counts() 

issue_d  delinquency_tf
2007.0   0                    558
         1                     45
2008.0   0                   2146
         1                    247
2009.0   0                   4687
         1                    594
2010.0   0                  11050
         1                   1487
2011.0   0                  18424
         1                   3297
2012.0   0                  44723
         1                   8644
2013.0   0                 113787
         1                  21027
2014.0   0                 194060
         1                  41569
2015.0   0                 343653
         1                  77442
2016.0   0                 360707
         1                  73700
2017.0   0                 394914
         1                  48665
2018.0   0                 477544
         1                  17698
Name: count, dtype: int64

time: 29.2 s (started: 2026-06-24 20:06:11 -05:00)


In [14]:
year_summary = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    delinquencies=('delinquency_tf', 'sum'),
    defaults=('defaulted_tf', 'sum')
)

year_summary['delinquency_rate'] = year_summary['delinquencies'] / year_summary['total_loans']
year_summary['default_rate'] = year_summary['defaults'] / year_summary['total_loans']

year_summary['delinquency_rate'] = (year_summary['delinquency_rate'] * 100).round(2)
year_summary['default_rate'] = (year_summary['default_rate'] * 100).round(2)

print(year_summary)

         total_loans  delinquencies  defaults  delinquency_rate  default_rate
issue_d                                                                      
2007.0           603             45        45              7.46          7.46
2008.0          2393            247       247             10.32         10.32
2009.0          5281            594       594             11.25         11.25
2010.0         12537           1487      1487             11.86         11.86
2011.0         21721           3297      3297             15.18         15.18
2012.0         53367           8644      8644             16.20         16.20
2013.0        134814          21027     21024             15.60         15.59
2014.0        235629          41569     41162             17.64         17.47
2015.0        421095          77442     75804             18.39         18.00
2016.0        434407          73700     68252             16.97         15.71
2017.0        443579          48665     39169             10.97 

In [15]:
year_summary['delinquencies'] - year_summary['defaults']

issue_d
2007.0       0
2008.0       0
2009.0       0
2010.0       0
2011.0       0
2012.0       0
2013.0       3
2014.0     407
2015.0    1638
2016.0    5448
2017.0    9496
2018.0    8824
dtype: int64

time: 1.9 ms (started: 2026-06-24 20:09:50 -05:00)


In [16]:
status_cts

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: Int64

time: 1.91 ms (started: 2026-06-24 20:09:52 -05:00)


In [17]:
loan_status_by_year = dfa_data.groupby([dfa_data['issue_d'].dt.year, 'loan_status']).size()

print(loan_status_by_year)

issue_d  loan_status                                        
2007.0   Charged Off                                                45
         Does not meet the credit policy. Status:Charged Off       113
         Does not meet the credit policy. Status:Fully Paid        239
         Fully Paid                                                206
2008.0   Charged Off                                               247
         Does not meet the credit policy. Status:Charged Off       249
         Does not meet the credit policy. Status:Fully Paid        582
         Fully Paid                                               1315
2009.0   Charged Off                                               594
         Does not meet the credit policy. Status:Charged Off       129
         Does not meet the credit policy. Status:Fully Paid        436
         Fully Paid                                               4122
2010.0   Charged Off                                              1487
         Does no

In [18]:
vintage = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    charged_off=('loan_status', lambda x: (x == 'Charged Off').sum()),
    default=('loan_status', lambda x: (x == 'Default').sum()),
    late_31_120=('loan_status', lambda x: (x == 'Late (31-120 days)').sum()),
    fully_paid=('loan_status', lambda x: (x == 'Fully Paid').sum()),
    current=('loan_status', lambda x: (x == 'Current').sum())
)

vintage['delinquency_rate'] = (vintage['charged_off'] + vintage['late_31_120']) / vintage['total_loans'] * 100
vintage['default_rate'] = vintage['charged_off'] / vintage['total_loans'] * 100   # Most common definition

vintage = vintage.round(2)
print(vintage)

         total_loans  charged_off  default  late_31_120  fully_paid  current  \
issue_d                                                                        
2007.0           603           45        0            0         206        0   
2008.0          2393          247        0            0        1315        0   
2009.0          5281          594        0            0        4122        0   
2010.0         12537         1487        0            0       10049        0   
2011.0         21721         3297        0            0       18424        0   
2012.0         53367         8644        0            0       44723        0   
2013.0        134814        21024        0            3      113780        6   
2014.0        235629        41161        1          334      181941    11919   
2015.0        421095        75803        1         1359      299742    43299   
2016.0        434407        68242       10         4546      224853   134061   
2017.0        443579        39148       

### Quick reflection:

- Lending Club experienced explosive growth in loan volume starting in 2013, peaking in 2017–2018.
- 2007–2012 had lower volume but relatively clean performance.
- From 2013–2016, both delinquency and charge-off rates increased significantly. 
- "Default" status is rare in the data. Most serious outcomes appear as "Charged Off".
- Late payments grew substantially after 2013. Could be due to increasing borrower stress after 2008 recession, or Lending Club's willingness to take on riskier loans. 

##### Some questions I'm considering:

- Why are outright "Defaults" rare, while "Charged Off" numbers are much higher? What does this pattern signal about their business practices, collections, and underwriting strategy?
- What does the steady rise in late payment statuses imply about borrower behavior and portfolio health?
- What incentives might Lending Club have had for allowing more loans to reach "Charged Off" status rather than earlier default?

In [20]:
from scipy.stats import chi2_contingency, pearsonr, chi2
categorical_columns = ['addr_state', 'home_ownership', 'purpose', 'sub_grade', 'verification_status']

a = 0.05

for col in categorical_columns:
    p_contingency_table = pd.crosstab(dfa_data[col], dfa_data['delinquency_tf'])
    p_chi_sq, p_pval, p_dof, p_exp = chi2_contingency(p_contingency_table)
    p_crit_val = chi2.ppf((1-a), p_dof)
    print(f"Population ---- {col}:------- \nchi2 = {p_chi_sq:.2f} \ncritical_value = {p_crit_val:.2f} \nsignificance ratio = {p_chi_sq / p_crit_val:.2f} \np-value = {p_pval} \ndof = {p_dof} \n\n")

for col in categorical_columns:
    s_contingency_table = pd.crosstab(dfa[col], dfa['delinquency_tf'])
    s_chi_sq, s_pval, s_dof, s_exp = chi2_contingency(s_contingency_table)
    s_crit_val = chi2.ppf((1-a), s_dof)
    print(f"Sample ---- {col}:------- \nchi2 = {s_chi_sq:.2f} \ncritical_value = {s_crit_val:.2f} \nsignificance ratio = {s_chi_sq / s_crit_val:.2f} \np-value = {s_pval} \ndof = {s_dof} \n\n")

Population ---- addr_state:------- 
chi2 = 3505.47 
critical_value = 67.50 
significance ratio = 51.93 
p-value = 0.0 
dof = 50 


Population ---- home_ownership:------- 
chi2 = 6394.69 
critical_value = 11.07 
significance ratio = 577.63 
p-value = 0.0 
dof = 5 


Population ---- purpose:------- 
chi2 = 5775.37 
critical_value = 22.36 
significance ratio = 258.27 
p-value = 0.0 
dof = 13 


Population ---- sub_grade:------- 
chi2 = 122866.62 
critical_value = 48.60 
significance ratio = 2528.00 
p-value = 0.0 
dof = 34 


Population ---- verification_status:------- 
chi2 = 20631.48 
critical_value = 5.99 
significance ratio = 3443.48 
p-value = 0.0 
dof = 2 


Sample ---- addr_state:------- 
chi2 = 1008.95 
critical_value = 66.34 
significance ratio = 15.21 
p-value = 2.2187727469354612e-179 
dof = 49 


Sample ---- home_ownership:------- 
chi2 = 2230.12 
critical_value = 9.49 
significance ratio = 235.05 
p-value = 0.0 
dof = 4 


Sample ---- purpose:------- 
chi2 = 1291.21 
critical

### Chi-Square Test Results & Quick Observations

I ran Chi-square tests on several categorical variables to see how strongly they’re associated with `delinquency_tf`.

When I tested on the full dataset, almost every p-value came back as 0.0. That’s technically good news, but it doesn’t help me figure out which variables are actually the *strongest*. So I ran the tests on both the full population and different sample sizes to get a better sense of relative strength.

#### What is the Significance Ratio?

I’m still building my intuition around chi-square results. After some side research, I realized there isn’t one universal cutoff people use. So I started calculating a simple **Significance Ratio** to help me compare:

**Significance Ratio = chi² statistic ÷ critical value**

It should indicate **how many times stronger** the observed relationship is compared to what we’d expect if the variables were completely unrelated (pure chance).

**Example:**
- `sub_grade` had a chi² of 32,629 and a critical value around 66.34 → Significance Ratio ≈ **671x**

That’s an extremely strong signal.

#### Quick Observations:
- All the variables I tested are statistically significant.
- **`sub_grade`** is by far the strongest predictor — no surprise since it’s Lending Club’s own risk rating.
- I’m keeping an eye on overfitting risk, especially with high-cardinality variables like `addr_state` and `purpose`.

This step is helping me prioritize which categorical variables are actually worth keeping for the modeling phase.

In [26]:
numeric_cols = ['annual_inc', 'delinq_2yrs', 'dti', 'emp_length', 
                   'fico_range_high', 'fico_range_low', 'inq_last_6mths', 
                   'int_rate', 'loan_amnt', 'pub_rec', 'revol_bal', 
                   'revol_util', 'term']

p_corr_defaults = []
p_corr_delinquencies = []

for col in numeric_cols:
    p_1corr, p1_val = pearsonr(dfa_data[col], dfa_data['defaulted_tf'])
    p_corr_defaults.append({'variable': col,'correlation': p_1corr, 'pop_p_value': p1_val})
p_corr_dflt = pd.DataFrame(p_corr_defaults)
p_corr_dflt = p_corr_dflt.sort_values('correlation', ascending=False)

for col in numeric_cols:
    p_2corr, p2_val = pearsonr(dfa_data[col], dfa_data['delinquency_tf'])
    p_corr_delinquencies.append({'variable': col,'correlation': p_2corr, 'pop_p_value': p2_val})
p_corr_dlqcy = pd.DataFrame(p_corr_delinquencies)
p_corr_dlqcy = p_corr_dlqcy.sort_values('correlation', ascending=False)

print(f"POPULATION DEFAULTS:\n{p_corr_dflt}\n\n")
print(f"POPULATION DELINQUENCY:\n{p_corr_dlqcy}\n\n")




s_corr_defaults = []
s_corr_delinquencies = []

for col in numeric_cols:
    s_1corr, s1_val = pearsonr(dfa[col], dfa['defaulted_tf'])
    s_corr_defaults.append({'variable': col,'correlation': s_1corr, 'sample_p_value': s1_val})
s_corr_dflt = pd.DataFrame(s_corr_defaults)
s_corr_dflt = s_corr_dflt.sort_values('correlation', ascending=False)

for col in numeric_cols:
    s_2corr, s2_val = pearsonr(dfa[col], dfa['delinquency_tf'])
    s_corr_delinquencies.append({'variable': col,'correlation': s_2corr, 'sample_p_value': s2_val})
s_corr_dlqcy = pd.DataFrame(s_corr_delinquencies)
s_corr_dlqcy = s_corr_dlqcy.sort_values('correlation', ascending=False)


print(f"SAMPLE DEFAULTS:\n{s_corr_dflt}\n\n")
print(f"SAMPLE DELINQUENCY:\n{s_corr_dlqcy}\n\n")

POPULATION DEFAULTS:
           variable  correlation    pop_p_value
6    inq_last_6mths     0.083358   0.000000e+00
9           pub_rec     0.031747   0.000000e+00
8         loan_amnt     0.020703  9.063517e-213
1       delinq_2yrs     0.019044  2.359966e-180
3        emp_length    -0.014527  9.110257e-106
10        revol_bal    -0.020887  1.538957e-216
0        annual_inc    -0.024730  1.043113e-302
2               dti          NaN            NaN
4   fico_range_high          NaN            NaN
5    fico_range_low          NaN            NaN
7          int_rate          NaN            NaN
11       revol_util          NaN            NaN
12             term          NaN            NaN


POPULATION DELINQUENCY:
           variable  correlation     pop_pvalue
6    inq_last_6mths     0.081912   0.000000e+00
9           pub_rec     0.032786   0.000000e+00
8         loan_amnt     0.027195   0.000000e+00
1       delinq_2yrs     0.020641  1.629726e-211
3        emp_length    -0.017252  2.23359

### Thoughts and Notes

I decided to rerun the chi-square tests and pearson correlations for both the full population and 500k sample set. Hopefully these notes will help get me closer to some thoughtful feature selection for the modeling phase. Here’s where my head is at:

**Strongest signals so far:**
- `sub_grade` – Strongest categorical predictor. Seems obvious now since functionally, it is used to predict borrower patterns and history. 
- `inq_last_6mths` – Strong positive correlation with default/delinquency. Recent credit inquiries are a classic risk signal.
- `loan_amnt` – moderate positive correlation. Larger loans appear riskier.
- `annual_inc` and `emp_length` – negative correlations (protective factors).

**Variables I’m watching carefully:**
- `pub_rec` – statistically significant but feels “on the nose.” Concerned about potential overfitting.
- `dti` – returning NaN in correlations. Need to investigate missing values.
- Several other numeric columns also showing NaN – likely due to missing data that needs cleaning.

**Next steps I’m considering:**
- Start exploring interactions (e.g. short employment + high DTI).



In [42]:
# === DTI Investigation ===

print("=== DTI Data Quality Check ===")

# 1. How many missing values?
print(f"Missing dti values: {dfa_data['dti'].isna().sum():,}")
print(f"Percentage missing: {dfa_data['dti'].isna().mean()*100:.2f}%")

# 2. Data type
print(f"dti data type: {dfa_data['dti'].dtype}")

# 3. Sample of missing dti rows
print("\n------------------------DTI <NA>/NaN sample set----------------------------\n")
print(dfa_data[dfa_data['dti'].isna()].head(20)[['loan_amnt', 'annual_inc', 'dti', 'defaulted_tf','delinquency_tf', 'loan_status']])
print("\n----------------------------------------------------\n")
# 4. Default rate when dti is missing
missing_default_rate = dfa_data[dfa_data['dti'].isna()]['defaulted_tf'].mean()
missing_delinquent_rate = dfa_data[dfa_data['dti'].isna()]['delinquency_tf'].mean()
print(f"\nDefault rate of DTI <NA>/NaN: {missing_default_rate:.4f} ({missing_default_rate*100:.2f}%)")
print(f"\nDelinquent rate of DTI <NA>/NaN: {missing_delinquent_rate:.4f} ({missing_delinquent_rate*100:.2f}%)\n\n")

=== DTI Data Quality Check ===
Missing dti values: 1,744
Percentage missing: 0.08%
dti data type: Float64

------------------------DTI <NA>/NaN sample set----------------------------

                     loan_amnt  annual_inc   dti  defaulted_tf  \
__null_dask_index__                                              
18202                  20000.0         0.0  <NA>             0   
65620                   3700.0         0.0  <NA>             1   
258                    17000.0         0.0  <NA>             0   
1655                   10000.0         0.0  <NA>             0   
3403                   18200.0         0.0  <NA>             0   
4093                   19200.0         0.0  <NA>             1   
7231                   10000.0         0.0  <NA>             0   
7484                   12225.0         0.0  <NA>             0   
9748                   15000.0         0.0  <NA>             1   
11113                   5600.0         0.0  <NA>             0   
12850                   

#### Note
- Missing dti values only occur when annual_inc = 0. Filling them with 0 would be misleading. 
- Creating a separate dti_missing flag feels like it could complicate the analysis and create extra branches I have to manage. At the same time, I don’t want to drop the rows or exclude dti from testing entirely.

### My Judgement Calls & Independent Decisions so far:

Throughout this project, I’ve made several deliberate decisions that differed from AI recommendations. I’m documenting them here to track my own reasoning and bookmark my work for later.

**1. Handling Missing `dti` Values**  
AI suggested filling missing `dti` with 0 or creating a `dti_missing` flag.  
- I rejected both approaches. Filling with 0 would distort the meaning (implying no debt or infinite repayment ability). Creating an extra flag felt like it would add unnecessary complexity and new cohorts to manage. 

**2. Sample Size Strategy**  
AI suggested working primarily with smaller samples for speed.  
- I pushed to analyze the full `dfa_data` for the main analysis using samples only for representativeness testing and heavy operations. I wanted the most accurate benchmarks.

**3. Variable Definitions**  
- I chose to maintain both `delinquency_tf` (broader) and `defaulted_tf` (stricter) instead of collapsing them early. This gives me flexibility to analyze different severity levels.

#### Where I'm at now:
I'm trying to balance statistical cleanliness with my intuition on real world interpretations. I need to weigh whether excluding the <NA> dti rows will be valuable for more tests, and modeling, or if it's better to keep the full data set intact. 


In [ ]:
plt.figure(figsize=(10, 6))
dfa['loan_status'].value_counts().plot(kind='bar')

plt.title('Loan Status Distribution')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#The best way is to store the results in a dictionary or DataFrame while running the loop, so you can pull specific values later.

In [ ]:
from scipy.stats import chi2_contingency, chi2

categorical_columns = ['addr_state', 'home_ownership', 'purpose', 'sub_grade', 'verification_status']
alpha = 0.05

pval = {}   # ← Add this line

for col in categorical_columns:
    contingency_table = pd.crosstab(dfa_data[col], dfa_data['delinquency_tf'])
    chi_sq, p_value, dof, exp = chi2_contingency(contingency_table)
    crit_val = chi2.ppf(1 - alpha, dof)
    
    pval[col] = p_value   # ← Save the p-value
    
    print(f"{col}: chi2 = {chi_sq:.2f} | p = {p_value:.2e} | ratio = {chi_sq / crit_val:.2f}x")

# Now you can easily access any p-value:
print("\nSpecific p-values:")
print(f"addr_state p-value: {pval['addr_state']}")
print(f"sub_grade p-value:   {pval['sub_grade']}")